# 의도 분류 보조 LLM 평가 — Colab · Qwen3.5-27B

backend 분류는 **규칙이 먼저** 하고, 규칙이 약한 세 군데에서만 분류 보조 LLM(Qwen3.5-27B)에 물어요. 이 노트북은 세 자리를 모두 **규칙만 vs 규칙+LLM**으로 채점해요. 답변 생성은 안 해요.

```
질문 → ① 보안문서 판단: 규칙 → 노사·징계·회의 단어가 있으면 LLM이 회의록 필요 여부를 다시 판단
     → 역할 분기
        ├─ 임직원: 규칙 분류 → ② 규칙이 sensitive·refuse가 아니면 LLM이 놓친 민감·타인 조회 보완 + 계산 값
        └─ 인사팀: 규칙(민감 표시 → 특정인 대화 기록 refuse → 본인 질문 → 집계 도구 → out_of_scope → rule)
                   → ③ 집계 도구가 정해지면 그 도구의 기간·기준값을 LLM에 묻고, 규칙이 비워 둔 값만 채움
```

| 자리 | 정답 데이터 | 채점 코드 (backend) | 보는 지표 |
| --- | --- | --- | --- |
| ③ 인사팀 기간·기준값 | RAG `eval/golden/admin_golden_set.json` | `scripts/eval_admin_intent.py` | 인자 일치율, LLM이 고친·새로 틀린 인자 (의도·도구는 두 방식이 같음) |
| ② 임직원 민감·타인·계산 | backend `eval/intent/intent_cases.jsonl` | `scripts/eval_intent.py` (`compare`) | 정확도, sensitive·refuse 재현율·정밀도, 계산 종류·값 |
| ① 보안문서 판단 | backend `eval/conf_doc/conf_doc_cases.jsonl` (41문항) | `scripts/eval_conf_doc.py` | 재현율(놓침), 오탐 |

- 로컬에서는 `python scripts/eval_admin_intent.py --llm`, `python scripts/eval_intent.py --llm`, `python scripts/eval_conf_doc.py --llm`이에요.
- 임직원 평가는 few-shot 예시(`app/data/classify_examples.json`)와 같은 쌍의 문장을 두 방식 모두 빼고 채점해요.
- 실행: 런타임 → 런타임 유형 변경 → **A100(40GB↑) 또는 H100** → 위에서부터 차례로.


## 1. 설정

In [ ]:
import os, sys, json, time, hashlib, subprocess, shutil
from pathlib import Path
from datetime import datetime

ROOT = Path("/content/hr_chatbot")
ROOT.mkdir(parents=True, exist_ok=True)
CFG = {
    "backend_branch": "feat/llm-assist-classify",   # 평가할 backend 브랜치
    "rag_branch": "main",           # 골든셋이 든 RAG 저장소 브랜치
    "llm_model": "auto",            # auto: 75GB↑ Qwen/Qwen3.5-27B · 39GB↑ -FP8. 27B가 안 들어가면 멈춤 (9B로 바꾸지 않음)
    "served_name": "qwen3.5-27b",
    "llm_port": 8000,
    "max_model_len": 8192,
    "split": "all",                 # all / dev / test. 프롬프트를 고칠 때는 dev, 최종 비교는 test 한 번만
}

## 2. GPU 확인 · 모델 고르기

In [ ]:
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise RuntimeError("GPU가 없어요. 런타임 → 런타임 유형 변경 → A100 또는 H100으로 바꾸고 다시 실행하세요.")
lines = gpu.splitlines()
GPU_NAME, N_GPU = lines[0].split(",")[0].strip(), len(lines)
TOTAL_GB = sum(int(l.split(",")[1]) for l in lines) / 1024
if CFG["llm_model"] == "auto":
    if TOTAL_GB >= 75:   CFG["llm_model"] = "Qwen/Qwen3.5-27B"
    elif TOTAL_GB >= 39: CFG["llm_model"] = "Qwen/Qwen3.5-27B-FP8"
    else:
        raise RuntimeError(f"{GPU_NAME} {TOTAL_GB:.0f}GB로는 Qwen3.5-27B가 안 들어가요 (L4 등). A100(40GB↑) 또는 H100으로 바꾸세요.")
print(f"GPU {GPU_NAME} × {N_GPU}, {TOTAL_GB:.0f}GB → {CFG['llm_model']}")

## 3. vLLM 설치 · 모델 다운로드 (백그라운드)

시작해 두고 아래 4번 셀을 계속 실행하세요. 첫 실행은 다운로드로 20~40분 걸려요.

In [ ]:
VLLM_ENV = Path("/content/vllm-env")
SETUP_LOG = Path("/content/llm_setup.log")
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""      # 보안 비밀(열쇠 아이콘)에 있으면 다운로드가 빨라요. 없어도 됨
except Exception:
    pass
setup_script = f"""set -e
echo "[1/2] vLLM 설치 $(date)"
if [ ! -x {VLLM_ENV}/bin/vllm ]; then
  pip install -q uv
  rm -rf {VLLM_ENV}
  uv venv {VLLM_ENV} --python 3.12 -q
  uv pip install --python {VLLM_ENV}/bin/python -q "vllm>=0.17.0" --torch-backend=auto
fi
echo "[2/2] 모델 다운로드 {CFG['llm_model']} $(date)"
{"export HF_TOKEN=" + HF_TOKEN if HF_TOKEN else ""}
{VLLM_ENV}/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('{CFG['llm_model']}')"
echo "SETUP_DONE $(date)"
"""
Path("/content/llm_setup.sh").write_text(setup_script)
if "SETUP_DONE" not in (SETUP_LOG.read_text() if SETUP_LOG.exists() else ""):
    SETUP_PROC = subprocess.Popen(f"bash /content/llm_setup.sh > {SETUP_LOG} 2>&1", shell=True)
    print("백그라운드 시작. 진행 상황:", SETUP_LOG)
else:
    print("이미 설치·다운로드됨")

## 4. backend · 골든셋 받기 (GitHub)

In [ ]:
for repo, branch in (("backend", CFG["backend_branch"]), ("RAG", CFG["rag_branch"])):
    d = ROOT / repo
    if not d.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, f"https://github.com/Jaeyong-s-HR-Chatbot/{repo}.git", str(d)], check=True)
    print(f"{repo:8s}", subprocess.run(["git", "-C", str(d), "log", "-1", "--format=%h %ad %s", "--date=short"],
                                      capture_output=True, text=True).stdout.strip())
BACKEND, RAG = ROOT / "backend", ROOT / "RAG"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-dotenv", "httpx", "PyJWT", "openai",
                "langchain-core", "langchain-openai"], check=True)

# backend를 불러오기 전에 환경 설정: DB 없음, 대화 로그 저장 안 함, 분류 LLM은 아래에서 vLLM으로 연결
os.environ.update({"HRDB_URL": "", "CHAT_LOG_DB": "false", "LLM_BASE_URL": "", "CLASSIFY_LLM": "off",
                   "PSEUDO_KEY": hashlib.sha256(os.urandom(16)).hexdigest()})
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
from app import config as be_config                     # noqa: E402
from app.graph import llm_classify, admin_params_llm, nodes as be_nodes   # noqa: E402
import importlib.util
spec = importlib.util.spec_from_file_location("eval_admin_intent", BACKEND / "scripts" / "eval_admin_intent.py")
ev = importlib.util.module_from_spec(spec); spec.loader.exec_module(ev)
ITEMS, AS_OF = ev.load_items(RAG / "eval" / "golden" / "admin_golden_set.json")
print(f"인사팀 문항 {len(ITEMS)}개, 기준일 {AS_OF}")

## 5. ③ 인사팀 규칙만 채점 (LLM 없이, 바로 돌아요)

In [ ]:
from IPython.display import Markdown, display
RULE = ev.run(CFG["split"], items=ITEMS, as_of=AS_OF, use_llm=False)
display(Markdown(ev.markdown(RULE)))

## 6. vLLM 띄우기

In [ ]:
import urllib.request
LLM_LOG = Path("/content/vllm.log")
LLM_API_KEY = hashlib.sha256(os.urandom(16)).hexdigest()[:32]
LLM_BASE_URL = f"http://127.0.0.1:{CFG['llm_port']}/v1"

while "SETUP_DONE" not in SETUP_LOG.read_text():
    if "SETUP_PROC" in globals() and SETUP_PROC.poll() not in (None, 0):
        print(SETUP_LOG.read_text()[-3000:]); raise RuntimeError("vLLM 설치·다운로드 실패 (로그 위)")
    last = (SETUP_LOG.read_text().strip().splitlines() or ["시작 전"])[-1]
    print("설치·다운로드 기다리는 중…", last[:100]); time.sleep(30)

if "LLM_PROC" in globals() and LLM_PROC.poll() is None:      # 앞 실행에서 남은 vLLM 끄기
    LLM_PROC.terminate()
    try: LLM_PROC.wait(timeout=60)
    except Exception: LLM_PROC.kill()
subprocess.run("pkill -f 'vllm serve' || true", shell=True)
time.sleep(3)

help_text = subprocess.run([str(VLLM_ENV / "bin/vllm"), "serve", "--help=all"], capture_output=True, text=True).stdout
cmd = [str(VLLM_ENV / "bin/vllm"), "serve", CFG["llm_model"], "--served-model-name", CFG["served_name"],
       "--host", "127.0.0.1", "--port", str(CFG["llm_port"]), "--api-key", LLM_API_KEY,
       "--max-model-len", str(CFG["max_model_len"]), "--gpu-memory-utilization", "0.9",
       "--max-num-seqs", "16",           # Qwen3.5 Mamba 캐시 때문에 기본 256이면 A100 40GB에서 시작이 안 돼요
       "--reasoning-parser", "qwen3", "--trust-remote-code", "--limit-mm-per-prompt", json.dumps({"image": 0, "video": 0})]
if "--default-chat-template-kwargs" in help_text:
    cmd += ["--default-chat-template-kwargs", json.dumps({"enable_thinking": False})]
if N_GPU > 1:
    cmd += ["--tensor-parallel-size", str(N_GPU)]
print(" ".join(cmd).replace(LLM_API_KEY, "***"))
LLM_PROC = subprocess.Popen(cmd, stdout=open(LLM_LOG, "w"), stderr=subprocess.STDOUT,
                            env={**os.environ, **({"HF_TOKEN": HF_TOKEN} if HF_TOKEN else {})})
t0 = time.time()
while True:
    try:
        req = urllib.request.Request(f"{LLM_BASE_URL}/models", headers={"Authorization": f"Bearer {LLM_API_KEY}"})
        json.loads(urllib.request.urlopen(req, timeout=5).read())
        print(f"✅ vLLM 준비 완료 ({time.time() - t0:.0f}초)"); break
    except Exception:
        if LLM_PROC.poll() is not None:
            log = LLM_LOG.read_text(errors="replace").splitlines()
            keys = ("out of memory", "OutOfMemory", "Error:", "ValueError", "not supported", "No available memory", "max_model_len", "CUDA")
            print("── 원인 후보 ──"); print("\n".join([l for l in log if any(k in l for k in keys)][-25:]) or "\n".join(log[-40:]))
            raise RuntimeError(f"vLLM이 시작하지 못했어요 (전체 로그: {LLM_LOG})")
        time.sleep(15)

## 7. backend 분류 LLM을 vLLM에 연결

backend는 분류 LLM을 Ollama 형식(`llm_classify.chat_json`)으로 불러요. 이 함수 하나만 vLLM(OpenAI 형식)으로 바꾸고, **프롬프트·JSON 스키마·응답 검증은 backend 그대로** 써요.

In [ ]:
from openai import OpenAI
LLM = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY, timeout=300, max_retries=2)
be_config.CLASSIFY_MODEL = CFG["served_name"]
CALLS = {"n": 0, "fail": 0, "ms": []}


def _vllm_chat_json(system, user, schema, num_predict=256):
    CALLS["n"] += 1
    t = time.time()
    try:
        r = LLM.chat.completions.create(
            model=CFG["served_name"],
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0, max_tokens=num_predict,
            response_format={"type": "json_schema", "json_schema": {"name": "out", "schema": schema}},
            extra_body={"chat_template_kwargs": {"enable_thinking": False}})
        return be_nodes._clean(r.choices[0].message.content or "")
    except Exception:
        CALLS["fail"] += 1
        raise ValueError("vLLM 요청 실패")          # backend가 잡아서 규칙 값으로 대체
    finally:
        CALLS["ms"].append((time.time() - t) * 1000)


llm_classify.chat_json = _vllm_chat_json
for q, tool in (("요즘 회사 그만둔 사람 어느 팀에서 많이 나갔어?", "retirements_by_dept"),
                ("올해 휴가 제일 많이 남은 사람 20명만", "leave_remaining_list")):
    t = time.time()
    r = admin_params_llm.extract(q, tool, AS_OF)
    print(f"{time.time() - t:5.1f}초 | {q} → {tool} {r}")

## 8. ③ 인사팀 규칙 + LLM(기간·기준값) 채점

In [ ]:
CALLS.update(n=0, fail=0, ms=[])
LLM_RES = ev.run(CFG["split"], items=ITEMS, as_of=AS_OF, use_llm=True)
display(Markdown("### 규칙만\n\n" + ev.markdown(RULE)))
display(Markdown("### 규칙 + LLM\n\n" + ev.markdown(LLM_RES)))
ms = CALLS["ms"]
print(f"LLM 호출 {CALLS['n']}번, 실패 {CALLS['fail']}번, 평균 {sum(ms) / len(ms):.0f}ms" if ms else "LLM 호출 없음")

### LLM이 고친 인자 · 새로 틀린 인자

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 70)
diff = []
for x, y in zip(RULE["rows"], LLM_RES["rows"]):
    if x["pred_params"] != y["pred_params"]:
        diff.append({"id": x["id"], "question": x["question"], "tool": x["pred_tool"], "정답": x["gold_params"],
                     "규칙": x["pred_params"], "규칙+LLM": y["pred_params"],
                     "결과": "고침" if y["params_ok"] and not x["params_ok"] else "새로 틀림" if x["params_ok"] and not y["params_ok"] else "그대로"})
df = pd.DataFrame(diff)
print(df["결과"].value_counts().to_dict() if len(df) else "LLM이 바꾼 인자 없음")
display(df)

### split별 · 틀린 문항

In [ ]:
display(pd.DataFrame([{"방식": name, "split": sp, **{k: v for k, v in s.items() if k != "라벨별"}}
                      for name, res in (("rule", RULE), ("rule+LLM", LLM_RES)) for sp, s in res["by_split"].items()]))
impl = set(admin_params_llm.admin_tools.ADMIN_TOOLS)
wrong = [x for x in LLM_RES["rows"] if not x["intent_ok"] or (x["gold_tool"] in impl and (x["tool_ok"] is False or x["params_ok"] is False))]
print(f"틀린 문항 {len(wrong)}개 (의도가 틀렸거나, backend에 있는 도구인데 도구·인자가 틀림)")
display(pd.DataFrame(wrong)[["id", "split", "question", "gold_intent", "pred_intent", "gold_tool", "pred_tool", "gold_params", "pred_params"]] if wrong else "없음")

## 9. ② 임직원 의도 분류 (규칙 vs 규칙+LLM)

규칙이 sensitive·refuse로 정하지 않은 문장에서만 LLM을 불러요. LLM은 **sensitive·refuse 판정**과 **계산 값**만 바꿀 수 있고, 나머지 의도·조회 도구는 규칙 그대로예요.


In [ ]:
spec = importlib.util.spec_from_file_location("eval_intent", BACKEND / "scripts" / "eval_intent.py")
ei = importlib.util.module_from_spec(spec); spec.loader.exec_module(ei)
be_config.USE_FAKE_LLM = False          # 분류 LLM 켜기 (요청은 위에서 바꿔 끼운 vLLM으로 감)
CALLS.update(n=0, fail=0, ms=[])
EMP = ei.compare(CFG["split"])
display(Markdown(EMP["summary"]))
ms = CALLS["ms"]
print(f"LLM 호출 {CALLS['n']}번, 실패 {CALLS['fail']}번, 평균 {sum(ms) / len(ms):.0f}ms" if ms else "LLM 호출 없음")


In [ ]:
emp_changed = pd.DataFrame(EMP["changed"])
print(emp_changed["결과"].value_counts().to_dict() if len(emp_changed) else "LLM이 바꾼 문장 없음")
display(emp_changed)
display(Markdown(EMP["detail"]))


## 10. ① 보안문서 판단 (규칙 vs 규칙+LLM)

노사·노조·교섭·협의회·징계·처분·위원회·회의 같은 단어가 있는 질문만 LLM에 물어요. 권한 확인은 따로(`conf_retriever`) 해서, 여기서 틀려도 기밀이 새지는 않아요. 놓치면 일반 규정 답변으로 가고, 오탐이면 권한 없는 직원이 규정 질문에 거부 안내를 받아요.


In [ ]:
spec = importlib.util.spec_from_file_location("eval_conf_doc", BACKEND / "scripts" / "eval_conf_doc.py")
ec = importlib.util.module_from_spec(spec); spec.loader.exec_module(ec)
CALLS.update(n=0, fail=0, ms=[])
CONF = {"규칙": ec.run(False), "규칙+LLM": ec.run(True)}
display(pd.DataFrame([{"방식": k, "정확도": round(r["정확도"], 3), "재현율": round(r["재현율"], 3),
                       "놓침": f"{len(r['놓침'])}/{r['n_pos']}", "문서 틀림": len(r["문서 틀림"]),
                       "오탐": f"{len(r['오탐'])}/{r['n_neg']}", "평균 ms": round(r["평균 ms"])} for k, r in CONF.items()]))
conf_wrong = [{"방식": k, "종류": kind, "id": x["id"], "question": x["question"], "정답": x["doc"], "예측": x["pred"]}
              for k, r in CONF.items() for kind in ("놓침", "문서 틀림", "오탐") for x in r[kind]]
display(pd.DataFrame(conf_wrong) if conf_wrong else "틀린 문항 없음")
print(f"LLM 호출 {CALLS['n']}번, 실패 {CALLS['fail']}번")


## 11. 결과 저장

`/content/admin_intent_results/`에 저장하고 zip으로 내려받아요. Google Drive가 연결되면 Drive에도 복사해요.


In [ ]:
out_dir = Path("/content/admin_intent_results") / f"classify_llm_{datetime.now():%Y%m%d_%H%M}_{CFG['served_name']}_{CFG['split']}"
ev.save(RULE, out_dir / "rule")
ev.save(LLM_RES, out_dir / "rule_llm")
(out_dir / "employee").mkdir(parents=True, exist_ok=True)
(out_dir / "employee" / "compare.md").write_text(EMP["report"], encoding="utf-8")
emp_changed.to_csv(out_dir / "employee" / "changed.csv", index=False, encoding="utf-8-sig")
(out_dir / "conf_doc").mkdir(parents=True, exist_ok=True)
(out_dir / "conf_doc" / "rows.json").write_text(json.dumps({k: r["rows"] for k, r in CONF.items()}, ensure_ascii=False, indent=1), encoding="utf-8")
(out_dir / "cfg.json").write_text(json.dumps({**CFG, "llm_calls": CALLS["n"], "llm_failures": CALLS["fail"]}, ensure_ascii=False, indent=1), encoding="utf-8")
print("저장:", out_dir)
zip_path = shutil.make_archive(str(out_dir), "zip", out_dir)
from google.colab import files
files.download(zip_path)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = Path("/content/drive/MyDrive/hr_chatbot_eval") / out_dir.name
    shutil.copytree(out_dir, dst, dirs_exist_ok=True)
    print("Drive에도 저장:", dst)
except Exception as e:
    print(f"Drive 연결 안 됨 ({type(e).__name__}) → 내려받은 zip을 쓰세요")